## Golden Valley Subsidence, CA USA (2021)

The InSAR.dev ecosystem, the PyGMTSAR InSAR library, the Geomed3D geophysical inversion library, and N-Cube 3D/4D GIS data visualization (among others) are open-source projects I develop in my free time.

I hold a Master’s degree in STEM, specializing in radio physics. In 2004, I received first prize in the All-Russian Physics Competition for significant results in forward and inverse modeling for nonlinear optics and holography. These skills are also applicable to modeling gravity, magnetic, and thermal fields, as well as satellite interferometry processing.

With 20 years of experience as a data scientist and software developer, I have contributed to scientific and industrial development through government contracts, university projects, and work with companies including LG Corp and Google Inc.

You can support my work on Patreon, where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform Upwork.

You can support my work on [Patreon](https://www.patreon.com/pechnikov), where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform [Upwork](https://www.upwork.com) or reach out to me directly.

### Resources
- Google Colab Pro notebooks and articles on [Patreon](https://www.patreon.com/pechnikov),
- Google Colab notebooks on [GitHub](https://github.com),
- Docker Images on [DockerHub](https://hub.docker.com),
- Geological Models on [YouTube](https://www.youtube.com),
- VR/AR Geological Models on [GitHub](https://github.com),
- Live updates and announcements on [LinkedIn](https://www.linkedin.com/in/alexey-pechnikov/).

© Alexey Pechnikov, 2025

$\large\color{blue}{\text{Hint: Use menu Cell} \to \text{Run All or Runtime} \to \text{Run All}}$
$\large\color{blue}{\text{(depending on your IDE) to execute the entire notebook.}}$

# Stage 1. InSAR.dev-PyGMTSAR: A Python package for InSAR pre-processing

Convert Sentinel-1 SLC data into a geocoded, cloud-ready Zarr dataset. The output can be hosted on GitHub or any file storage as a set of files or a single ZIP archive.

## Google Colab Installation

Install InSAR.dev Python libraries

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !{sys.executable} -m pip install -q pyvista xvfbwrapper jupyter_bokeh vtk panel
    #trame trame-vuetify trame-vtk
    from google.colab import output
    output.enable_custom_widget_manager()
    # install the exact commit (no pip cache)
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev_toolkit"
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev_pygmtsar"
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev"

## Import Python Modules

In [ ]:
import numpy as np
import geopandas as gpd
import shapely
import json
import matplotlib.pyplot as plt
%matplotlib inline
# setup dark theme
from insardev.UI import UI
UI('dark')

In [ ]:
# print versions
from insardev_pygmtsar import __version__ as pygmtsar_version
from insardev_toolkit import __version__ as toolkit_version
print("insardev_pygmtsar version:", pygmtsar_version)
print("insardev_toolkit version:", toolkit_version)
# import modules to be used in the notebook
from insardev_pygmtsar import S1
from insardev_toolkit import EOF, ASF, Tiles, XYZTiles

## Specify Sentinel-1 SLC Bursts and Area

### Descending Orbit Bursts

https://search.asf.alaska.edu/#/?polygon=POLYGON((-118.4711%2034.3916,-118.47%2034.3915,-118.4688%2034.3916,-118.467%2034.3921,-118.4658%2034.3924,-118.4646%2034.3928,-118.4629%2034.3936,-118.461%2034.3942,-118.4578%2034.3939,-118.4595%2034.399,-118.46%2034.3994,-118.4631%2034.3976,-118.4674%2034.3952,-118.469%2034.394,-118.4699%2034.3934,-118.471%2034.3926,-118.4717%2034.3918,-118.4711%2034.3916))&start=2021-01-05T17:00:00Z&end=2021-12-20T16:59:59Z&resultsLoaded=true&granule=S1_151226_IW3_20211220T135244_VV_571D-BURST&flightDirs=Descending&zoom=9.794&center=-118.575,34.110&path=71-71&frame=480-480&dataset=SENTINEL-1%20BURSTS&polarizations=VV

In [ ]:
# Specify bursts to download
BURSTS = """
S1_151226_IW3_20211220T135244_VV_571D-BURST
S1_151226_IW3_20211208T135245_VV_5BB3-BURST
S1_151226_IW3_20211126T135245_VV_F7BC-BURST
S1_151226_IW3_20211114T135246_VV_582A-BURST
S1_151226_IW3_20211102T135246_VV_06BA-BURST
S1_151226_IW3_20211021T135246_VV_AC92-BURST
S1_151226_IW3_20211009T135246_VV_8A1C-BURST
S1_151226_IW3_20210927T135246_VV_080A-BURST
S1_151226_IW3_20210915T135245_VV_B51B-BURST
S1_151226_IW3_20210903T135245_VV_6CBC-BURST
S1_151226_IW3_20210822T135244_VV_7922-BURST
S1_151226_IW3_20210810T135244_VV_A317-BURST
S1_151226_IW3_20210729T135243_VV_1A1B-BURST
S1_151226_IW3_20210717T135242_VV_0ABE-BURST
S1_151226_IW3_20210705T135242_VV_0F30-BURST
S1_151226_IW3_20210623T135241_VV_7E2B-BURST
S1_151226_IW3_20210611T135240_VV_D775-BURST
S1_151226_IW3_20210530T135240_VV_A70D-BURST
S1_151226_IW3_20210518T135239_VV_660C-BURST
S1_151226_IW3_20210506T135238_VV_2C81-BURST
S1_151226_IW3_20210424T135238_VV_82DC-BURST
S1_151226_IW3_20210412T135237_VV_7DB6-BURST
S1_151226_IW3_20210331T135237_VV_91F1-BURST
S1_151226_IW3_20210319T135236_VV_34F9-BURST
S1_151226_IW3_20210307T135236_VV_0C1D-BURST
S1_151226_IW3_20210223T135236_VV_883C-BURST
S1_151226_IW3_20210211T135237_VV_DAE0-BURST
S1_151226_IW3_20210130T135237_VV_30E3-BURST
S1_151226_IW3_20210118T135237_VV_5815-BURST
S1_151226_IW3_20210106T135238_VV_D0B4-BURST
"""
BURSTS = list(filter(None, BURSTS.split('\n')))
print (f'Bursts defined: {len(BURSTS)}')

## Specify Directories

In [ ]:
# directory where Sentinel-1 bursts and orbits will be downloaded
DATADIR = 'golden_valley_data'
# path to the downloaded DEM file
DEM = f'{DATADIR}/dem.nc'
ZARRDIR = 'golden_valley_zarr'

## Download and Unpack Datasets

### Enter Your ASF User and Password

If the data directory is empty or doesn't exist, you'll need to download Sentinel-1 scenes from the Alaska Satellite Facility (ASF) datastore. Use your Earthdata Login credentials. If you don't have an Earthdata Login, you can create one at https://urs.earthdata.nasa.gov//users/new

You can also use pre-existing SLC scenes stored on your Google Drive, or you can copy them using a direct public link from iCloud Drive.

The credentials below are available at the time the notebook is validated.

In [ ]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf_username = 'GoogleColab2023'
asf_password = 'GoogleColab_2023'

In [ ]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf = ASF(asf_username, asf_password)
print(asf.download(DATADIR, BURSTS))

In [ ]:
# read the notices printed below carefully
s1 = S1(DATADIR)
s1.to_dataframe().to_file(f"{DATADIR}/s1.geojson", driver="GeoJSON")
#df = gpd.read_file("s1.geojson")
#s1.to_dataframe()

In [ ]:
# scan the data directory for Sentinel-1 SLC bursts and download related orbits
EOF().download(DATADIR, s1.to_dataframe())
#s1.to_dataframe()

In [ ]:
# download Copernicus Global DEM 1 arc-second
Tiles().download_dem(s1.to_dataframe(), provider='GLO', filename=DEM)[::4,::4].plot.imshow(cmap='cividis')

## Preprocess Sentinel-1 SLC to Geocoded Zarr

In [ ]:
# scan SLC bursts with DEM
s1 = S1(DATADIR, DEM=DEM)
s1.to_dataframe()

In [ ]:
# descending orbit bursts: preview
s1.plot(ref='2021-07-05')

In [ ]:
# load SLC bursts and orbits with DEM and transform to ZARR
s1.transform(ZARRDIR, ref='2021-07-05')

# Stage 2. InSAR.dev: A Python package for InSAR processing

Load Sentinel-1 geocoded, cloud-ready Zarr dataset locally or hosted on GitHub or any file storage as a set of files or a single ZIP archive.

## Import Python Modules

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# setup dark theme
from insardev.UI import UI
UI('dark')

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !{sys.executable} -m pip install -q --no-cache-dir \
        "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev"

In [ ]:
# print versions
from insardev import __version__ as insardev_version
from insardev_toolkit import __version__ as toolkit_version
print("insardev version:", insardev_version)
print("insardev_toolkit version:", toolkit_version)
# import modules to be used in the notebook
from insardev import Stack, BatchUnit
# data downloader
from insardev_toolkit.HTTP import unzip

In [ ]:
# common data science libraries
import xarray as xr
import numpy as np
import pandas as pd
import rioxarray as rio
import scipy.stats

In [ ]:
# 3D plotting modules
import pyvista as pv
# white background
#pv.set_plot_theme("document")
pv.set_plot_theme("dark")
from IPython.display import display, HTML
if 'google.colab' in sys.modules:
    import panel
    panel.extension(comms='ipywidgets')
    panel.extension('vtk')
# 2D interactive map modules
import json
import matplotlib.colors
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps, basemap_to_tiles

## Specify Directories

In [ ]:
ZARRDIR = 'golden_valley_zarr'
WORKDIR = 'golden_valley_workdir'

## Specify AOI

In [ ]:
geojson = '''
{
  "type": "Feature",
  "properties": {},
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [-118.4711, 34.3916],
        [-118.47, 34.3915],
        [-118.4688, 34.3916],
        [-118.467, 34.3921],
        [-118.4658, 34.3924],
        [-118.4646, 34.3928],
        [-118.4629, 34.3936],
        [-118.461, 34.3942],
        [-118.4578, 34.3939],
        [-118.4595, 34.399],
        [-118.46, 34.3994],
        [-118.4631, 34.3976],
        [-118.4674, 34.3952],
        [-118.469, 34.394],
        [-118.4699, 34.3934],
        [-118.471, 34.3926],
        [-118.4717, 34.3918],
        [-118.4711, 34.3916]
      ]
    ]
  }
}
'''
AOI = gpd.GeoDataFrame.from_features([json.loads(geojson)]).set_crs('EPSG:4326').to_crs('EPSG:32611')
AOI

In [ ]:
# subsidence point from https://earthdaily.com/blog/sentinel-1-targeted-analysis
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [-118.4652, 34.3952]
  },
  "properties": {}
}
'''
POI = gpd.GeoDataFrame.from_features([json.loads(geojson)]).set_crs('EPSG:4326').to_crs('EPSG:32611')
# compensate different interpolation between PyGMTSAR and InSAR.dev
POI.geometry = POI.translate(xoff=15, yoff=15)
POI

In [ ]:
# reference point from https://earthdaily.com/blog/sentinel-1-targeted-analysis
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [-118.4629,34.3946]
  },
  "properties": {}
}
'''
POI0 = gpd.GeoDataFrame.from_features([json.loads(geojson)]).set_crs('EPSG:4326').to_crs('EPSG:32611')
# compensate different interpolation between PyGMTSAR and InSAR.dev
POI0.geometry = POI0.translate(yoff=15, xoff=15)
POI0

## Download and Unpack Datasets

We can process datasets hosted on GitHub, Zenodo, or other platforms in a single workflow.

In [ ]:
# example downloading command
#unzip("https://zenodo.org/records/15347694/files/Türkiye_Elevation-40x10-004.zip", ZARRDIR)

## Run Local Dask Cluster

Launch a Dask cluster for local or distributed multicore computing. This makes it possible to process terabyte-scale Sentinel-1 SLC datasets even on an Apple MacBook Air with 16 GB of RAM.

In [ ]:
# simple Dask initialization
if 'client' in globals():
    client.shutdown()
    client.close()
# For stubborn memory, reset CUDA device
import torch
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
if torch.backends.mps.is_available():
    # Don't call synchronize() - it can hang if there was an error!
    # Just empty cache and collect garbage
    torch.mps.empty_cache()
    gc.collect()
    # twice helps sometimes
    torch.mps.empty_cache()
import logging
logging.getLogger('distributed').setLevel(logging.CRITICAL)
from dask.distributed import Client
# set threads_per_worker=1 to prevent the error 'CUDA out of memory' on GPU instances
client = Client(silence_logs='CRITICAL', threads_per_worker=1, resources={'gpu': 1})
client

## Load Geocoded Bursts

In [ ]:
stack = Stack().load(ZARRDIR)
stack

In [ ]:
stack.plot(cmap='autumn', alpha=0.15)
# download the basemap adding the buffer to cover the area of interest after reprojecting
gmap = XYZTiles().download_googlesatellite(stack.to_dataframe().buffer(5_000), zoom=10, fill_value=0)
gmap.plot.imshow(ax=plt.gca(), zorder=-1, add_labels=False)
plt.gca().set_title(f'Sentinel-1 Burst Footprints');

## Crop Bursts by AOI

In [ ]:
# enlarge AOI by buffer to ensure proper coverage
bounds = AOI.buffer(2500).total_bounds
stack = stack.sel(x=slice(bounds[0], bounds[2]), y=slice(bounds[1], bounds[3]))
stack

## Compute Baseline

In [ ]:
# 57 pairs with 24 days temporal baseline
baseline = stack.baseline(days=24)
baseline

In [ ]:
baseline.plot();

## Multi-Look Analysis

Multilooking applies anti-aliasing filtering ('wavelength' parameter) and downscaling (or less accurate decimation) to improve signal-to-noise ratio and reduce raster size. Alternatively, we can skip multilooking when signal-to-noise ratio is good.

### Compute Interferogram and Correlation

In [ ]:
# 30m antialiasing Gaussian filter
mintf, mcorr = stack.phasediff(baseline.tolist(), wavelength=30).compute()
mintf

In [ ]:
mintf.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, caption='Interferometric Phase, [rad]');

In [ ]:
mcorr.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, caption='Correlation Coefficient');

### 2D Unwrap Interferograms

In [ ]:
mphase = stack.unwrap2d(mintf, mcorr).compute()
mphase

In [ ]:
mphase.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Phase, [rad]');

### 2D Detrend Unwrapped Phase

In [ ]:
mphase_trend = stack.trend2d(mphase, mcorr, stack.transform()[['azi','rng','ele']])
mphase_trend

In [ ]:
mphase_trend.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, vmin=-np.pi, vmax=np.pi, alpha=0.8, caption='Trend Phase, [rad]');

In [ ]:
mphase_detrend = mphase - mphase_trend
mphase_detrend

In [ ]:
mphase_detrend.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, vmin=-np.pi, vmax=np.pi, alpha=0.8, caption='Detrended Phase, [rad]');

### Compute Accumulated 1D Phase

In [ ]:
mphase_acc = stack.lstsq(mphase_detrend, mcorr).compute()
mphase_acc

In [ ]:
mphase_acc.isel(date=slice(None, None, 4)).plot(rows=3, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Accumulated Phase, [rad]');

### Convert Phase to Line-of-Sight (LOS) Displacement

In [ ]:
mdisplacement_los = stack.displacement_los(mphase_acc)
mdisplacement_los

In [ ]:
mdisplacement_los.isel(date=slice(None, None, 4)).plot(rows=3, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Displacement, [m]');

### Compute LOS Velocity

In [ ]:
mvelocity, mdisp0 = mdisplacement_los.velocity()
mvelocity

In [ ]:
mvelocity.plot(quantile=[0.01, 0.99], alpha=0.8, symmetrical=True, caption='Velocity, [m/year]');

## Single-Look Analysis

Single-look interferograms are noisier and harder to unwrap. By removing the trend detected from multi-looked unwrapped phases, we can reduce phase variations and improve single-look unwrapping at original resolution. With a high-quality, fast unwrapper, it is also possible to combine 2D + 1D unwrapping on single-look interferograms.

### Compute Interferogram and Correlation

In [ ]:
sintf, scorr = stack.phasediff(baseline.tolist(), wavelength=30, phase=mphase_trend).compute()
sintf

In [ ]:
sintf.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, caption='Interferometric Phase, [rad]');

In [ ]:
scorr.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, caption='Correlation Coefficient');

### 1D Unwrap Interferograms

In [ ]:
sphase = stack.unwrap1d(sintf, scorr).compute()

In [ ]:
sphase.isel(pair=slice(None, None, 8)).plot(rows=2, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Phase, [rad]');

### Compute Accumulated 1D Phase

In [ ]:
sphase_acc = stack.lstsq(sphase, scorr)

In [ ]:
sphase_acc.isel(date=slice(None, None, 4)).plot(rows=3, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Accumulated Phase, [rad]');

### Convert Phase to Line-of-Sight (LOS) Displacement

In [ ]:
sdisplacement_los = stack.displacement_los(sphase_acc)

In [ ]:
sdisplacement_los.isel(date=slice(None, None, 4)).plot(rows=3, cols=4, size=2.5, quantile=[0.01, 0.99], alpha=0.8, caption='Displacement, [m]');

### Compute Velocity

In [ ]:
svelocity, sdisp0 = sdisplacement_los.velocity()
svelocity

In [ ]:
svelocity.plot(quantile=[0.01, 0.99], alpha=0.8, symmetrical=True, caption='Velocity, [m/year]');

## Multi-Look vs Single-Look Analysis Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mvel_data = 1000*mvelocity.mask(AOI).to_dataset()['VV']
svel_data = 1000*svelocity.mask(AOI).to_dataset()['VV']
vmin, vmax = -30, 30
for ax, data, title in zip(axes, [mvel_data, svel_data], ['Multi-Look Velocity (AOI), [mm/year]', 'Single-Look Velocity (AOI), [mm/year]']):
    im = data.plot.imshow(ax=ax, vmin=vmin, vmax=vmax, cmap='turbo',
                        alpha=0.8, add_colorbar=False, interpolation='none')
    POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
    POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI0')
    ax.legend(loc='upper left')
    ax.set_title(title)
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='mm/year')
plt.show()

In [ ]:
plt.figure(figsize=(12, 4), dpi=150)

# Get displacement data as xarray Datasets
mdisp_ds = mdisplacement_los.to_dataset()
sdisp_ds = sdisplacement_los.to_dataset()

# Get velocity and intercept data
mvel_ds = mvelocity.to_dataset()
mint_ds = mdisp0.to_dataset()
svel_ds = svelocity.to_dataset()
sint_ds = sdisp0.to_dataset()

# POI (subsidence point) - red/yellow
x, y = POI.geometry[0].coords[0]
mdisp_pixel = mdisp_ds.VV.sel(y=y, x=x, method='nearest')
sdisp_pixel = sdisp_ds.VV.sel(y=y, x=x, method='nearest')
mvel_poi = float(mvel_ds.VV.sel(y=y, x=x, method='nearest').values)
mint_poi = float(mint_ds.VV.sel(y=y, x=x, method='nearest').values)
svel_poi = float(svel_ds.VV.sel(y=y, x=x, method='nearest').values)
sint_poi = float(sint_ds.VV.sel(y=y, x=x, method='nearest').values)

# Pearson correlation for POI
corr_poi, _ = scipy.stats.pearsonr(mdisp_pixel.values, sdisp_pixel.values)

# Time in years
dates = pd.to_datetime(mdisp_pixel.date.values)
t_years = (dates - dates[0]).total_seconds() / (365.25 * 24 * 3600)

# Displacement
plt.plot(dates, 1000*mdisp_pixel, c='r', lw=1, label='Multi-Look Displacement POI')
plt.plot(dates, 1000*sdisp_pixel, c='y', lw=1, label='Single-Look Displacement POI')

# Velocity lines
plt.plot(dates, 1000*(mint_poi + mvel_poi * t_years), c='r', lw=2, ls=':', label=f'Multi-Look Velocity POI ({1000*mvel_poi:.0f} mm/yr)')
plt.plot(dates, 1000*(sint_poi + svel_poi * t_years), c='y', lw=2, ls=':', label=f'Single-Look Velocity POI ({1000*svel_poi:.0f} mm/yr)')

# POI0 (reference point) - blue/cyan
x0, y0 = POI0.geometry[0].coords[0]
mdisp_pixel0 = mdisp_ds.VV.sel(y=y0, x=x0, method='nearest')
sdisp_pixel0 = sdisp_ds.VV.sel(y=y0, x=x0, method='nearest')
mvel_poi0 = float(mvel_ds.VV.sel(y=y0, x=x0, method='nearest').values)
mint_poi0 = float(mint_ds.VV.sel(y=y0, x=x0, method='nearest').values)
svel_poi0 = float(svel_ds.VV.sel(y=y0, x=x0, method='nearest').values)
sint_poi0 = float(sint_ds.VV.sel(y=y0, x=x0, method='nearest').values)

# Pearson correlation for POI0
corr_poi0, _ = scipy.stats.pearsonr(mdisp_pixel0.values, sdisp_pixel0.values)

# Displacement
plt.plot(dates, 1000*mdisp_pixel0, c='b', lw=2, label='Multi-Look Displacement POI0')
plt.plot(dates, 1000*sdisp_pixel0, c='c', lw=1, label='Single-Look Displacement POI0')

# Velocity lines
plt.plot(dates, 1000*(mint_poi0 + mvel_poi0 * t_years), c='b', lw=2, ls=':', label=f'Multi-Look Velocity POI0 ({1000*mvel_poi0:.0f} mm/yr)')
plt.plot(dates, 1000*(sint_poi0 + svel_poi0 * t_years), c='c', lw=2, ls=':', label=f'Single-Look Velocity POI0 ({1000*svel_poi0:.0f} mm/yr)')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=10)
plt.title(f'LOS Displacement and Velocity, 2021\nPearson Correlation: POI={corr_poi:.2f}, POI0={corr_poi0:.2f}', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.grid()
plt.tight_layout()
plt.show()

## 3D Interactive Map

In [ ]:
gmap = XYZTiles().download(AOI.buffer(500), 15)
gmap.plot.imshow()

In [ ]:
stack.crop(AOI.buffer(200)).to_vtk('gmap', data=None, overlay=gmap)

In [ ]:
stack.to_vtk('mvel', 1000*mvelocity.crop(AOI))
stack.to_vtk('svel', 1000*svelocity.crop(AOI))

In [ ]:
plotter = pv.Plotter(shape=(1, 2), notebook=True)
axes = pv.Axes(show_actor=True, actor_scale=2.0, line_width=5)

vtk_map = pv.read('gmap.vtk').scale([1, 1, 1.98]).rotate_z(135)

plotter.subplot(0, 0)
vtk_grid = pv.read('mvel/VV.vtk').scale([1, 1, 2]).rotate_z(135)
plotter.add_mesh(vtk_map, scalars='colors', rgb=True, ambient=0.2)
plotter.add_mesh(vtk_grid, scalars='VV', ambient=0.5, opacity=0.5, cmap='turbo', clim=(-30,30), nan_opacity=0.1, nan_color='black',
                scalar_bar_args={'title': 'Multi-Look Velocity, mm/year', 'vertical': True})
bounds = vtk_map.bounds
center = vtk_map.center
# move camera closer by reducing distance (zoom in)
distance = vtk_map.length / 2
plotter.camera.position = (center[0] + distance, center[1] + distance, center[2] + distance * 0.5)
plotter.camera.focal_point = center
plotter.camera.azimuth = 110
plotter.camera.elevation = -5

plotter.subplot(0, 1)
vtk_grid = pv.read('svel/VV.vtk').scale([1, 1, 2]).rotate_z(135)
plotter.add_mesh(vtk_map, scalars='colors', rgb=True, ambient=0.2)
plotter.add_mesh(vtk_grid, scalars='VV', ambient=0.5, opacity=0.5, cmap='turbo', clim=(-30,30), nan_opacity=0.1, nan_color='black',
                scalar_bar_args={'title': 'Single-Look Velocity, mm/year', 'vertical': True})
bounds = vtk_map.bounds
center = vtk_map.center
# move camera closer by reducing distance (zoom in)
distance = vtk_map.length / 2
plotter.camera.position = (center[0] + distance, center[1] + distance, center[2] + distance * 0.5)
plotter.camera.focal_point = center
plotter.camera.azimuth = 110
plotter.camera.elevation = -5

#plotter.show_axes()
plot = plotter.show(screenshot='3D Velocity.png', jupyter_backend='panel' if 'google.colab' in sys.modules else 'client', return_viewer=True)
if 'google.colab' in sys.modules:
    plot = panel.panel(
        plotter.render_window, orientation_widget=plotter.renderer.axes_enabled,
        enable_keybindings=False, sizing_mode='stretch_width', min_height=600
    )
display(HTML('<h1 style="text-align:center; margin-bottom:0;">LOS Displacement Velocity on DEM, mm/year</h1>'))
plot